In [1]:
!pip install xgboost optuna --quiet

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import warnings, os, pickle
from pathlib import Path
import yfinance as yf

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")

In [3]:
np.random.seed(20160101)
os.makedirs("results/mean_rev_final", exist_ok=True)

# DJIA 30 Mean Reversion — Asymmetric Long-Short  (FINAL)

**Version.** This is the final production version of the mean-reversion sleeve for the ICM. Differences from `MeanRev_AsymLS_Pt1.ipynb`:

* **Optuna trials bumped from 50 to 80** for search-effort parity with the momentum sleeve (`MomBased-Pt2-updated.ipynb`).
* **Four standalone chart artifacts** matching the momentum notebook's plot inventory: validation performance, feature importance, rebalance weights heatmap, Optuna search landscape.
* **Per-rebalance weight logging** added to `run_backtest` to power the weight-heatmap chart.

**Strategy thesis.** Equities drift upward over time (positive equity risk premium). Short positions carry a structural headwind + unbounded-loss tail risk. This strategy accepts that asymmetry directly in sizing:

* **Long leg = 100% of capital** — full-weight into oversold bouncers.
* **Short leg = 20–40% of capital** (Optuna-tuned) — smaller, diversified bet on overbought pullbacks.
* **Net exposure ≈ +60–80% long, gross ≈ 120–140%.**

**Architecture.** Same scaffold as the momentum sleeve — PIT constituent map, 3-state regime classifier, XGBoost cross-sectional ranker, ridge portfolio optimizer, Optuna search — with three swaps: reversal features (5-day horizon), asymmetric long-short book, Sortino objective.

### Stage 1 â€” Data Collection (PIT constituent map)

In [4]:
# PIT-tracked DJIA 30 constituent map (verbatim from MomBased_Pt2).
# WBA excluded due to known yfinance data gaps; UTX was renamed to RTX before study start.

_BASELINE_APR2016 = {
    "AAPL", "AXP", "BA",  "CAT",  "CSCO", "CVX",  "DD",
    "DIS",  "GE",  "GS",  "HD",   "IBM",  "INTC", "JNJ",
    "JPM",  "KO",  "MCD", "MMM",  "MRK",  "MSFT", "NKE",
    "PFE",  "PG",  "RTX", "TRV",  "UNH",  "V",    "VZ",
    "WMT",  "XOM",
}

_CHANGES = [
    ("2018-06-26", ["WBA"],              ["GE"]),
    ("2019-04-02", ["DOW"],              ["DD"]),
    ("2020-04-06", ["RTX"],              ["UTX"]),
    ("2020-08-31", ["AMGN","CRM","HON"], ["XOM","PFE","RTX"]),
    ("2024-02-26", ["AMZN","SHW"],       ["WBA","INTC"]),
    ("2024-11-01", ["NVDA"],             ["DOW"]),
]


def fetch_prices_volumes(start_date="2016-04-01", end_date="2026-04-18"):
    """Download adjusted close AND volume â€” volumes needed for volume_z feature."""
    all_tickers = set(_BASELINE_APR2016)
    for _, added, removed in _CHANGES:
        all_tickers.update(added)
    all_tickers -= {"WBA", "UTX"}
    all_tickers = sorted(all_tickers)

    raw     = yf.download(all_tickers, start=start_date, end=end_date,
                          auto_adjust=True, progress=False)
    prices  = raw["Close"].ffill(limit=10)
    volumes = raw["Volume"].ffill(limit=10)
    prices  = prices.dropna(axis=1, how="all")
    volumes = volumes.reindex(columns=prices.columns)
    return prices, volumes


def get_constituents_on_date(date):
    """Return the exact DJIA 30 constituents on a given date."""
    constituents = set(_BASELINE_APR2016)
    for change_date_str, added, removed in _CHANGES:
        if date >= pd.Timestamp(change_date_str):
            constituents.update(added)
            constituents -= set(removed)
    constituents -= {"WBA", "UTX"}
    return constituents

### Stage 2 â€” Reversal Feature Engineering

Five short-horizon reversal features:

| Feature | Definition | Oversold signal |
|---|---|---|
| `ret_5` | trailing 5-day return | most negative |
| `rsi_2` | Connors-style short RSI | near 0 |
| `dist_bb` | (price âˆ’ 20d SMA) / (2 Ã— 20d std), Bollinger z-score | negative |
| `vol_20` | annualized 20-day realized vol | high (reversal edge strongest in high-dispersion names) |
| `volume_z` | today's volume z-score vs 20-day mean/std | high abs value (panic / frenzy marker) |

**Target**: 5-day forward return, cross-sectionally ranked to [0, 1]. **Non-overlap rule**: training rows subsampled every 5 days to avoid label overlap leakage on the 5-day horizon.

In [5]:
FEATURE_COLS = ["ret_5", "rsi_2", "dist_bb", "vol_20", "volume_z"]


def compute_rsi(prices_arr, period):
    """RSI from the last (period+1) closing prices. Same helper as MomBased_Pt2."""
    delta  = np.diff(prices_arr[-(period + 1):])
    gains  = delta[delta > 0].sum() / period
    losses = -delta[delta < 0].sum() / period
    if losses == 0:
        return 100.0
    return 100.0 - 100.0 / (1.0 + gains / losses)


def compute_features_reversal(
    prices,
    volumes,
    rsi_period=2,
    bb_window=20,
    vol_window=20,
    volume_window=20,
    fwd_days=5,
):
    """
    Build daily cross-sectional reversal feature snapshots.
    Returns long-format DataFrame with one row per (date, ticker).
    """
    log_rets = np.log(prices / prices.shift(1))
    min_day = max(fwd_days, bb_window, vol_window, volume_window, rsi_period + 1) + 2
    records = []

    for di in range(min_day, len(prices) - fwd_days):
        date   = prices.index[di]
        fwd_di = di + fwd_days

        pit_tickers = get_constituents_on_date(date)
        pit_tickers = [t for t in pit_tickers if t in prices.columns]

        row_data = []
        for t in pit_tickers:
            px  = prices[t].values
            vol = volumes[t].values

            if np.isnan(px[di]) or np.isnan(px[fwd_di]):
                continue
            if di - max(bb_window, vol_window, volume_window) < 0:
                continue

            p_back = px[di - fwd_days]
            if np.isnan(p_back) or p_back <= 0:
                continue
            ret_5 = px[di] / p_back - 1.0

            if di - rsi_period < 0:
                continue
            try:
                rsi = compute_rsi(px[: di + 1], rsi_period)
            except Exception:
                continue

            px_win = px[di - bb_window + 1 : di + 1]
            if np.any(np.isnan(px_win)):
                continue
            sma = px_win.mean()
            sd  = px_win.std()
            if sd <= 0:
                continue
            dist_bb = (px[di] - sma) / (2.0 * sd)

            r_slice = log_rets[t].iloc[di - vol_window + 1 : di + 1].values
            if np.any(np.isnan(r_slice)):
                continue
            vol_20 = float(np.std(r_slice)) * np.sqrt(252.0)

            v_win = vol[di - volume_window + 1 : di + 1]
            if np.any(np.isnan(v_win)) or v_win.std() <= 0:
                continue
            volume_z = (vol[di] - v_win.mean()) / v_win.std()

            fwd = px[fwd_di] / px[di] - 1.0

            row_data.append({
                "date": date, "ticker": t,
                "ret_5": ret_5, "rsi_2": rsi, "dist_bb": dist_bb,
                "vol_20": vol_20, "volume_z": volume_z,
                "fwd_ret": fwd,
            })

        df_row = pd.DataFrame(row_data)
        if df_row.empty:
            continue
        df_row["fwd_rank"] = df_row["fwd_ret"].rank(pct=True, na_option="keep")
        df_row = df_row.dropna(subset=["fwd_ret", "fwd_rank"] + FEATURE_COLS)
        if len(df_row) >= 5:
            records.append(df_row)

    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

### Stage 3 â€” Regime Classifier (verbatim from Pt2)

Three market states are derived from the rolling vol and drawdown of an equal-weight PIT index:

* **Trending** â€” normal markets (~75% of days)
* **Volatile** â€” elevated vol or moderate drawdown (~18%)
* **Crash** â€” extreme vol or deep drawdown (~7%)

The regime state drives the asymmetric sizing logic in the backtest.

In [6]:
def classify_regimes(
    prices,
    vol_window=20,
    dd_window=60,
    vol_crash=0.28,
    vol_vol=0.17,
    dd_crash=-0.13,
    dd_vol=-0.07,
):
    pit_index = []
    for date in prices.index:
        members = get_constituents_on_date(date)
        valid = [t for t in members if t in prices.columns and pd.notna(prices.loc[date, t])]
        pit_index.append(prices.loc[date, valid].mean() if valid else np.nan)
    mkt = pd.Series(pit_index, index=prices.index).ffill()
    log_rets = np.log(mkt / mkt.shift(1))
    roll_vol = log_rets.rolling(vol_window).std() * np.sqrt(252)
    roll_pk  = mkt.rolling(dd_window).max()
    roll_dd  = (mkt - roll_pk) / roll_pk

    regimes = pd.Series("trending", index=prices.index, dtype=str)
    regimes[(roll_vol > vol_vol)   | (roll_dd < dd_vol)]   = "volatile"
    regimes[(roll_vol > vol_crash) | (roll_dd < dd_crash)] = "crash"
    return regimes

### Stage 4 â€” XGBoost Cross-Sectional Ranker

Predicts cross-sectional forward rank from the 5 reversal features. Same architecture as Pt2 â€” regression on `fwd_rank` (0â€“1 percentile). Higher predicted rank â†’ expected bouncer; lower predicted rank â†’ expected pullback.

In [7]:
def train_xgboost(train_df, max_depth=3):
    clean = train_df[FEATURE_COLS + ["fwd_rank"]].replace([np.inf, -np.inf], np.nan).dropna()
    X = clean[FEATURE_COLS].values.astype(np.float32)
    y = clean["fwd_rank"].values.astype(np.float32)
    if len(X) == 0:
        raise ValueError("Training data is empty after cleaning.")

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=max_depth,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1.0,
        min_child_weight=5,
        reg_lambda=1.0,
        reg_alpha=0.1,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    model.fit(X, y)
    return model


def predict_scores(model, feat_df):
    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    return model.predict(X)

### Stage 5 â€” Asymmetric Long-Short Portfolio Construction

Two independent ridge solves:

**Long leg:**
$$ \max \sum \text{score}_i \cdot w_i - \lambda_{\text{long}} \|w\|^2 \quad \text{s.t.} \quad \sum w_i = \text{gross}_L, \; 0 \le w_i \le \text{cap}_L $$

**Short leg:**
$$ \max \sum (1 - \text{score}_i) \cdot w_i - \lambda_{\text{short}} \|w\|^2 \quad \text{s.t.} \quad \sum w_i = \text{gross}_S, \; 0 \le w_i \le \text{cap}_S $$

Short leg uses `(1 âˆ’ score)` as strength because we want more weight on names XGBoost expects to fall. Weights are then signed (+1 for long leg, âˆ’1 for short leg) and concatenated.

`Î»_short > Î»_long` by design â€” a single mis-identified short is a squeeze risk (unbounded loss); a single mis-identified long is bounded at âˆ’100%. Higher Î» on the short side forces breadth.

**Per-name caps**: 12% long, 6% short (halved) â€” another layer of short-tail defense.

In [8]:
def ridge_optimize_signed(
    scores, tickers,
    ridge_lambda=0.5,
    top_n=10,
    gross=1.0,
    direction=+1,
    pick="top",
    per_name_cap=None,
):
    """
    Ridge-regularised weight vector, scaled to target gross, signed by direction.
    """
    order = np.argsort(scores)
    sel_idx = order[::-1][:top_n] if pick == "top" else order[:top_n]

    sel_tickers = [tickers[i] for i in sel_idx]
    sel_scores  = scores[sel_idx]

    # For short leg, flip so low fwd_rank predictions get high "strength"
    strength = (1.0 - sel_scores) if direction < 0 else sel_scores

    raw_w = np.maximum(0.0, strength) / (2.0 * ridge_lambda)
    total = raw_w.sum()
    w_mag = raw_w / total if total > 0 else np.full(top_n, 1.0 / top_n)

    w_mag = w_mag * gross

    if per_name_cap is not None:
        w_mag = np.minimum(w_mag, per_name_cap)
        if w_mag.sum() > 0 and w_mag.sum() < gross:
            residual = gross - w_mag.sum()
            unused = np.maximum(0.0, per_name_cap - w_mag)
            if unused.sum() > 0:
                w_mag = w_mag + residual * unused / unused.sum()

    signed = w_mag * direction
    return dict(zip(sel_tickers, signed))


def asymmetric_long_short(
    scores, tickers,
    lambda_long=0.5, lambda_short=2.0,
    top_n_long=15, top_n_short=8,
    gross_long=1.0, gross_short=0.4,
    cap_long=0.12, cap_short=0.06,
    size_long=1.0, size_short=1.0,
):
    """
    Build asymmetric long-short book.
      LONG  leg: top_n_long picks (highest predicted fwd_rank)   = expected bouncers
      SHORT leg: top_n_short picks (lowest  predicted fwd_rank)  = expected pullbacks
    Regime multipliers (size_long, size_short) scale each leg's gross exposure.
    """
    w_long = ridge_optimize_signed(
        scores, tickers,
        ridge_lambda=lambda_long, top_n=top_n_long,
        gross=gross_long * size_long, direction=+1, pick="top",
        per_name_cap=cap_long,
    )
    w_short = ridge_optimize_signed(
        scores, tickers,
        ridge_lambda=lambda_short, top_n=top_n_short,
        gross=gross_short * size_short, direction=-1, pick="bottom",
        per_name_cap=cap_short,
    )
    combined = dict(w_long)
    for t, w in w_short.items():
        combined[t] = w   # short leg wins if a ticker somehow lands in both
    return combined

### Stage 6 â€” Backtest with Signed Weights, Borrow Cost, Regime-Aware Sizing

**Regime sizing** â€” asymmetric across regimes:

| Regime | size_long | size_short | Rationale |
|---|---|---|---|
| Trending (~75%) | 1.00Ã— | **0.25Ã—** | Long leg earns through drift; short leg fights the momentum factor |
| Volatile (~18%) | 1.00Ã— | **1.00Ã—** | High dispersion = reversal's home regime |
| Crash (~7%)    | 1.00Ã— | **0.00Ã—** | Shorts killed entirely â€” squeeze risk spikes in bear-market rallies |

**Costs:**
* Turnover: 10 bps Ã— `|Î”w|.sum()` on every rebalance (same as Pt2)
* Borrow: 25 bps/yr Ã— short notional, accrued daily

**Splits:** train 2016â€“20, val 2020â€“22 (Optuna sees this), test 2022â€“26 (held out).

In [9]:
REGIME_SIZING = {
    "trending": (1.00, 0.25),   # long full, short trimmed (fighting momentum factor)
    "volatile": (1.00, 1.00),   # both full (dispersion opportunity)
    "crash":    (1.00, 0.00),   # long-only (squeeze defense)
}

TURNOVER_BPS      = 10.0
BORROW_BPS_ANNUAL = 25.0


def sortino_ratio(daily_rets, mar=0.0):
    """Annualized Sortino ratio from daily returns."""
    excess = daily_rets - mar / 252.0
    downside = np.minimum(0.0, excess)
    dd_dev = np.sqrt(np.mean(downside ** 2))
    if dd_dev == 0:
        return float("inf") if excess.mean() > 0 else 0.0
    return (excess.mean() / dd_dev) * np.sqrt(252.0)


def run_backtest(
    prices, features_df, regimes,
    rsi_period=2, rebal_freq=1,
    lambda_long=0.5, lambda_short=2.0,
    top_n_long=15, top_n_short=8,
    gross_long=1.0, gross_short=0.4,
    cap_long=0.12, cap_short=0.06,
    mode="val",
):
    rebal_days   = rebal_freq * 5
    unique_dates = np.sort(features_df["date"].unique())

    train_cutoff = pd.Timestamp("2020-01-01")
    val_cutoff   = pd.Timestamp("2022-01-01")

    train_dates = unique_dates[unique_dates <  train_cutoff]
    val_dates   = unique_dates[(unique_dates >= train_cutoff) & (unique_dates < val_cutoff)]
    test_dates  = unique_dates[unique_dates >= val_cutoff]

    eval_dates = val_dates if mode == "val" else test_dates

    if len(train_dates) < 20 or len(eval_dates) < 20:
        return {"sharpe": -99.0, "sortino": -99.0, "cagr": -99.0, "max_dd": -99.0}

    # Non-overlap rule: subsample training to every 5th day for 5-day forward target
    train_df_full = features_df[features_df["date"].isin(train_dates)]
    keep_train_dates = np.sort(train_df_full["date"].unique())[::5]
    train_df = train_df_full[train_df_full["date"].isin(keep_train_dates)]

    model   = train_xgboost(train_df)
    test_df = features_df[features_df["date"].isin(eval_dates)]

    test_start  = pd.Timestamp(eval_dates[0])
    test_prices = prices[prices.index >= test_start].copy()

    curr_weights = {}
    all_weights_log = []  # per-rebalance weight snapshots
    strat_val = 100.0
    bench_val = 100.0
    long_val  = 100.0
    short_val = 100.0
    peak = 100.0
    max_dd = 0.0
    kills_short = 0
    trim_trending = 0
    last_rebal = -rebal_days

    strat_curve, bench_curve, date_index, regime_log = [], [], [], []
    long_leg_curve, short_leg_curve = [], []

    for di in range(1, len(test_prices)):
        date = test_prices.index[di]

        pit_tickers = get_constituents_on_date(date)
        pit_tickers = [t for t in pit_tickers if t in test_prices.columns]

        daily_rets = {}
        for t in pit_tickers:
            p0 = test_prices[t].iloc[di - 1]
            p1 = test_prices[t].iloc[di]
            if pd.notna(p0) and pd.notna(p1) and p0 > 0:
                daily_rets[t] = p1 / p0 - 1.0

        reg = regimes.get(date, "trending")
        regime_log.append(reg)

        if di - last_rebal >= rebal_days:
            last_rebal = di
            avail = test_df[test_df["date"] <= date]
            if not avail.empty:
                snap_date = avail["date"].max()
                snap = (
                    test_df[(test_df["date"] == snap_date) &
                            (test_df["ticker"].isin(pit_tickers))]
                    .set_index("ticker")
                    .dropna(subset=FEATURE_COLS)
                )

                if len(snap) >= top_n_long + top_n_short:
                    scores_arr = predict_scores(model, snap.reset_index())
                    tickers_avail = snap.index.tolist()

                    size_long_mult, size_short_mult = REGIME_SIZING[reg]
                    if reg == "crash":
                        kills_short += 1
                    if reg == "trending":
                        trim_trending += 1

                    new_weights = asymmetric_long_short(
                        scores_arr, tickers_avail,
                        lambda_long=lambda_long, lambda_short=lambda_short,
                        top_n_long=top_n_long, top_n_short=top_n_short,
                        gross_long=gross_long, gross_short=gross_short,
                        cap_long=cap_long, cap_short=cap_short,
                        size_long=size_long_mult, size_short=size_short_mult,
                    )

                    # Turnover cost on |Î”w|
                    all_t = set(new_weights) | set(curr_weights)
                    turnover = sum(abs(new_weights.get(t, 0.0) - curr_weights.get(t, 0.0))
                                   for t in all_t)
                    cost = turnover * TURNOVER_BPS / 10000.0
                    strat_val *= (1.0 - cost)
                    curr_weights = new_weights
                    all_weights_log.append({"date": date, "weights": dict(new_weights)})

        # P&L â€” signed weights
        strat_ret = sum(curr_weights.get(t, 0.0) * daily_rets.get(t, 0.0)
                        for t in curr_weights if t in daily_rets)
        long_ret  = sum(w * daily_rets.get(t, 0.0)
                        for t, w in curr_weights.items()
                        if w > 0 and t in daily_rets)
        short_ret = sum(w * daily_rets.get(t, 0.0)
                        for t, w in curr_weights.items()
                        if w < 0 and t in daily_rets)

        # Daily borrow cost on short notional
        short_notional    = sum(abs(w) for w in curr_weights.values() if w < 0)
        borrow_cost_daily = short_notional * (BORROW_BPS_ANNUAL / 10000.0) / 252.0
        strat_ret -= borrow_cost_daily
        short_ret -= borrow_cost_daily

        bench_ret = float(np.mean([daily_rets[t] for t in pit_tickers if t in daily_rets]))

        strat_val *= (1.0 + strat_ret)
        bench_val *= (1.0 + bench_ret)
        long_val  *= (1.0 + long_ret)
        short_val *= (1.0 + short_ret)
        peak      = max(peak, strat_val)
        max_dd    = min(max_dd, (strat_val - peak) / peak)

        strat_curve.append(strat_val)
        bench_curve.append(bench_val)
        long_leg_curve.append(long_val)
        short_leg_curve.append(short_val)
        date_index.append(date)

    if len(strat_curve) < 50:
        return {"sharpe": -99.0, "sortino": -99.0, "cagr": -99.0, "max_dd": -99.0}

    daily_rets_arr = np.diff(strat_curve) / np.array(strat_curve[:-1])
    n_years    = len(strat_curve) / 252.0
    cagr       = (strat_val / 100.0) ** (1.0 / n_years) - 1.0
    bench_cagr = (bench_val / 100.0) ** (1.0 / n_years) - 1.0
    sharpe  = ((daily_rets_arr.mean() / daily_rets_arr.std()) * np.sqrt(252.0)
               if daily_rets_arr.std() > 0 else 0.0)
    sortino = sortino_ratio(daily_rets_arr)

    feat_imp = dict(zip(FEATURE_COLS, model.feature_importances_))

    return {
        "sharpe":          round(float(sharpe),  4),
        "sortino":         round(float(sortino), 4),
        "cagr":            round(float(cagr * 100), 2),
        "bench_cagr":      round(float(bench_cagr * 100), 2),
        "max_dd":          round(float(max_dd * 100), 2),
        "cum_ret":         round(strat_val - 100.0, 2),
        "kills_short":     kills_short,
        "trim_trending":   trim_trending,
        "strat_curve":     strat_curve,
        "bench_curve":     bench_curve,
        "long_leg_curve":  long_leg_curve,
        "short_leg_curve": short_leg_curve,
        "date_index":      date_index,
        "regime_log":      regime_log,
        "feat_imp":        feat_imp,
        "model":           model,
        "weights":         curr_weights,
    }

### Stage 7 â€” Optuna Hyperparameter Search (Sortino objective)

Seven-dim search over:

| Param | Range | Notes |
|---|---|---|
| `rsi_period` | [2, 3, 5, 7] | Connors zone |
| `rebal_freq` | [1, 2, 3] weeks | 5d signal half-life |
| `lambda_long` | log-uniform [0.1, 3.0] | moderate regularization |
| `lambda_short` | log-uniform [0.5, 10.0] | aggressive â€” force breadth on short leg |
| `top_n_long` | [10, 15, 20] | wide long book = bankruptcy-proof |
| `top_n_short` | [5, 8, 12] | narrow short book = less squeeze surface |
| `gross_short` | [0.20, 0.30, 0.40, 0.50] | the asymmetry knob |

**Objective**: Sortino ratio (downside-only vol in denominator). Reversal returns are left-skewed; Ïƒ in standard Sharpe under-penalizes the falling-knife tail.

In [10]:
N_TRIALS = 80


def run_optuna(prices, volumes, regimes, n_trials=N_TRIALS, seed=42):
    feat_cache = {}

    def objective(trial):
        rsi_period   = trial.suggest_categorical("rsi_period",   [2, 3, 5, 7])
        rebal_freq   = trial.suggest_categorical("rebal_freq",   [1, 2, 3])
        lambda_long  = trial.suggest_float(      "lambda_long",  0.1, 3.0, log=True)
        lambda_short = trial.suggest_float(      "lambda_short", 0.5, 10.0, log=True)
        top_n_long   = trial.suggest_categorical("top_n_long",   [10, 15, 20])
        top_n_short  = trial.suggest_categorical("top_n_short",  [5, 8, 12])
        gross_short  = trial.suggest_categorical("gross_short",  [0.20, 0.30, 0.40, 0.50])

        if rsi_period not in feat_cache:
            feat_cache[rsi_period] = compute_features_reversal(
                prices, volumes, rsi_period=rsi_period, fwd_days=5)

        result = run_backtest(
            prices, feat_cache[rsi_period], regimes,
            rsi_period=rsi_period, rebal_freq=rebal_freq,
            lambda_long=lambda_long, lambda_short=lambda_short,
            top_n_long=top_n_long, top_n_short=top_n_short,
            gross_long=1.0, gross_short=gross_short,
            cap_long=0.12, cap_short=0.06,
            mode="val",
        )
        s = result["sortino"]
        return s if np.isfinite(s) else -99.0

    sampler = TPESampler(seed=seed)
    study   = optuna.create_study(direction="maximize", sampler=sampler,
                                  study_name="mean_rev_asym_ls")

    print(f"\n{'='*60}\n  Optuna TPE search  â€”  {n_trials} trials (Sortino)\n{'='*60}\n")

    def cb(study, trial):
        if trial.number % 5 == 0 or trial.number == n_trials - 1:
            best = study.best_value if study.best_trial else float("nan")
            v    = trial.value if trial.value is not None else float("nan")
            print(f"  Trial {trial.number+1:3d}/{n_trials}  Sortino={v:+.3f}  (best={best:+.3f})")

    study.optimize(objective, n_trials=n_trials, callbacks=[cb])

    best = study.best_params
    print(f"\n  BEST TRIAL #{study.best_trial.number + 1}  Sortino = {study.best_value:.4f}")
    for k, v in best.items():
        print(f"    {k:15s}: {v}")
    return study, best

### Stage 8 â€” Dashboard Plot

In [11]:
DARK = "#0a0c0f"; SURFACE = "#111418"; BORDER = "#232830"
TEXT = "#e2e8f0"; MUTED = "#8896a8"
GREEN = "#22c55e"; RED = "#ef4444"; AMBER = "#f59e0b"
BLUE = "#60a5fa"; PURPLE = "#a78bfa"; CYAN = "#22d3ee"

plt.rcParams.update({
    "figure.facecolor": DARK,    "axes.facecolor":  SURFACE,
    "axes.edgecolor":   BORDER,  "axes.labelcolor": MUTED,
    "xtick.color":      MUTED,   "ytick.color":     MUTED,
    "text.color":       TEXT,    "grid.color":      BORDER,
    "grid.linewidth":   0.5,     "font.family":     "monospace",
    "axes.titlecolor":  TEXT,    "axes.titlesize":  10,
    "axes.titleweight": "bold",
})


def plot_backtest(result, best_params, save_path):
    fig = plt.figure(figsize=(16, 12), facecolor=DARK)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35,
                            left=0.07, right=0.97, top=0.92, bottom=0.07)

    dates   = pd.DatetimeIndex(result["date_index"])
    strat   = np.array(result["strat_curve"])
    bench   = np.array(result["bench_curve"])
    long_c  = np.array(result["long_leg_curve"])
    short_c = np.array(result["short_leg_curve"])
    regimes = result["regime_log"]

    reg_colors = {"trending": GREEN, "volatile": AMBER, "crash": RED}

    # Panel 1: cumulative performance
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(dates, strat, color=CYAN, lw=2.0,
             label=f"Strategy   CAGR {result['cagr']:.1f}%", zorder=3)
    ax1.plot(dates, bench, color=BLUE, lw=1.4, ls="--", alpha=0.7,
             label=f"Benchmark  CAGR {result['bench_cagr']:.1f}%", zorder=2)

    prev_reg, seg_start = regimes[0], dates[0]
    for i in range(1, len(dates)):
        if regimes[i] != prev_reg or i == len(dates) - 1:
            ax1.axvspan(seg_start, dates[i], alpha=0.09,
                        color=reg_colors[prev_reg], zorder=1)
            seg_start = dates[i]; prev_reg = regimes[i]

    legend_patches = [
        Patch(facecolor=GREEN, alpha=0.4, label="Trending (L 1.0x, S 0.25x)"),
        Patch(facecolor=AMBER, alpha=0.4, label="Volatile (L 1.0x, S 1.0x)"),
        Patch(facecolor=RED,   alpha=0.4, label="Crash (L 1.0x, S OFF)"),
    ]
    ax1.legend(handles=[*ax1.get_lines(), *legend_patches],
               loc="upper left", fontsize=8.5,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT, ncol=2)
    ax1.set_title("CUMULATIVE PERFORMANCE  (shaded = regime)", pad=8)
    ax1.set_ylabel("Portfolio value  (base = 100)")
    ax1.grid(True, alpha=0.3)

    # Panel 2: drawdown
    ax2 = fig.add_subplot(gs[1, :2])
    peak_arr = np.maximum.accumulate(strat)
    dd_arr   = (strat - peak_arr) / peak_arr * 100.0
    ax2.fill_between(dates, dd_arr, 0, color=RED, alpha=0.45)
    ax2.plot(dates, dd_arr, color=RED, lw=0.8)
    ax2.set_title(f"DRAWDOWN  (max {result['max_dd']:.1f}%)")
    ax2.set_ylabel("%")
    ax2.grid(True, alpha=0.3)

    # Panel 3: feature importance
    ax3 = fig.add_subplot(gs[1, 2])
    feat_imp = result["feat_imp"]
    labels = ["ret_5", f"rsi_{best_params['rsi_period']}", "dist_bb",
              "vol_20", "volume_z"]
    vals = [float(feat_imp.get(f, 0.0)) for f in FEATURE_COLS]
    total = sum(vals) if sum(vals) > 0 else 1.0
    vals_pct = np.array(vals) / total * 100.0
    colors_fi = [GREEN, BLUE, PURPLE, AMBER, CYAN]
    bars = ax3.barh(labels, vals_pct, color=colors_fi, height=0.6)
    for bar, v in zip(bars, vals_pct):
        ax3.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
                 f"{v:.1f}%", va="center", fontsize=9, color=TEXT)
    ax3.set_title("XGBOOST FEATURE IMPORTANCE")
    ax3.set_xlim(0, max(vals_pct) * 1.3)
    ax3.grid(True, alpha=0.3, axis="x")

    # Panel 4: leg attribution
    ax4 = fig.add_subplot(gs[2, :2])
    long_ret  = (long_c[-1]  / long_c[0]  - 1) * 100
    short_ret = (short_c[-1] / short_c[0] - 1) * 100
    ax4.plot(dates, long_c, color=GREEN, lw=1.6,
             label=f"Long leg   {long_ret:+.1f}%  (oversold bouncers)")
    ax4.plot(dates, short_c, color=RED, lw=1.6,
             label=f"Short leg  {short_ret:+.1f}%  (overbought pullbacks)")
    ax4.axhline(100.0, color=MUTED, ls=":", lw=0.8, alpha=0.6)
    ax4.set_title("LEG ATTRIBUTION  (each leg's standalone P&L)")
    ax4.set_ylabel("Leg value  (base = 100)")
    ax4.legend(loc="upper left", fontsize=8.5,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax4.grid(True, alpha=0.3)

    # Panel 5: metrics + best params
    ax5 = fig.add_subplot(gs[2, 2])
    ax5.axis("off")
    rows = [
        ("Ann. return",    f"{result['cagr']:.2f}%",       CYAN),
        ("Benchmark",      f"{result['bench_cagr']:.2f}%", BLUE),
        ("Sharpe ratio",   f"{result['sharpe']:.4f}",
         GREEN if result["sharpe"] > 0.5 else AMBER),
        ("Sortino ratio",  f"{result['sortino']:.4f}",
         GREEN if result["sortino"] > 0.7 else AMBER),
        ("Max drawdown",   f"{result['max_dd']:.2f}%",     RED),
        ("Cumul. return",  f"{result['cum_ret']:.1f}%",
         GREEN if result["cum_ret"] > 0 else RED),
        ("Short-kill days",f"{result['kills_short']}",     PURPLE),
        ("-- best params --", "", MUTED),
        ("RSI period",     f"{best_params['rsi_period']}d",TEXT),
        ("Rebal freq",     f"{best_params['rebal_freq']}w",TEXT),
        ("lambda long",    f"{best_params['lambda_long']:.3f}",  TEXT),
        ("lambda short",   f"{best_params['lambda_short']:.3f}", TEXT),
        ("Top-N long",     f"{best_params['top_n_long']}",  TEXT),
        ("Top-N short",    f"{best_params['top_n_short']}", TEXT),
        ("Gross short",    f"{best_params['gross_short']:.2f}",  TEXT),
    ]
    for i, (label, val, color) in enumerate(rows):
        y = 1.0 - i * 0.065
        ax5.text(0.02, y, label, transform=ax5.transAxes,
                 fontsize=9, color=MUTED, va="top")
        ax5.text(0.98, y, val, transform=ax5.transAxes,
                 fontsize=9, color=color, va="top", ha="right", fontweight="bold")

    fig.suptitle(
        "DOW 30 MEAN REVERSION  //  Asymmetric Long-Short  //  XGBoost + Ridge + Regime",
        fontsize=11.5, fontweight="bold", color=TEXT, y=0.97
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  Saved: {save_path}")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# STANDALONE PLOTS (match momentum notebook's chart inventory)
# ══════════════════════════════════════════════════════════════════════════════

def plot_validation_performance(
    prices, features_df, regimes, best_params,
    save_path="results/mean_rev_final/validation_performance.png",
):
    """Run backtest on val window with best params, plot cumulative + drawdown."""
    val_result = run_backtest(
        prices, features_df, regimes,
        rsi_period=best_params["rsi_period"],
        rebal_freq=best_params["rebal_freq"],
        lambda_long=best_params["lambda_long"],
        lambda_short=best_params["lambda_short"],
        top_n_long=best_params["top_n_long"],
        top_n_short=best_params["top_n_short"],
        gross_long=1.0,
        gross_short=best_params["gross_short"],
        cap_long=0.12, cap_short=0.06,
        mode="val",
    )
    dates = pd.DatetimeIndex(val_result["date_index"])
    strat = np.array(val_result["strat_curve"])
    bench = np.array(val_result["bench_curve"])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                    gridspec_kw={"height_ratios": [2, 1]},
                                    facecolor=DARK)
    ax1.plot(dates, strat, color=CYAN, lw=2.0,
             label=f"Strategy   CAGR {val_result['cagr']:.2f}%   Sharpe {val_result['sharpe']:.3f}")
    ax1.plot(dates, bench, color=BLUE, lw=1.4, ls="--", alpha=0.7,
             label=f"Benchmark  CAGR {val_result['bench_cagr']:.2f}%")
    ax1.set_title("VALIDATION PERFORMANCE  (2020-2022)", color=TEXT, pad=10,
                  fontweight="bold")
    ax1.set_ylabel("Portfolio value  (base = 100)", color=MUTED)
    ax1.legend(loc="upper left", fontsize=9,
               facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax1.grid(True, alpha=0.3)

    peak = np.maximum.accumulate(strat)
    dd = (strat - peak) / peak * 100.0
    ax2.fill_between(dates, dd, 0, color=RED, alpha=0.5)
    ax2.plot(dates, dd, color=RED, lw=0.8)
    ax2.set_title(f"DRAWDOWN  (max {val_result['max_dd']:.2f}%)", color=TEXT)
    ax2.set_ylabel("%", color=MUTED)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlabel("Date", color=MUTED)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  Saved: {save_path}")
    plt.close()
    return val_result


def plot_feature_importance_standalone(
    result, best_params,
    save_path="results/mean_rev_final/feature_importance.png",
):
    """Horizontal bar chart of XGBoost feature importances (test-period model)."""
    fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK)
    feat_imp = result["feat_imp"]
    labels = ["ret_5 (5-day return)",
              f"rsi_{best_params['rsi_period']} (short RSI)",
              "dist_bb (Bollinger z)",
              "vol_20 (20-day vol)",
              "volume_z (volume z)"]
    vals = [float(feat_imp.get(f, 0.0)) for f in FEATURE_COLS]
    total = sum(vals) if sum(vals) > 0 else 1.0
    vals_pct = np.array(vals) / total * 100.0
    colors_fi = [GREEN, BLUE, PURPLE, AMBER, CYAN]

    bars = ax.barh(labels, vals_pct, color=colors_fi, height=0.6)
    for bar, v in zip(bars, vals_pct):
        ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
                f"{v:.1f}%", va="center", fontsize=10, color=TEXT)
    ax.set_title("XGBOOST FEATURE IMPORTANCE (gain %)", color=TEXT, pad=10,
                 fontweight="bold")
    ax.set_xlim(0, max(vals_pct) * 1.3)
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  Saved: {save_path}")
    plt.close()


def plot_rebalance_weights(
    result,
    save_path="results/mean_rev_final/rebalance_weights.png",
    max_tickers=20,
):
    """Heatmap of position weights at each rebalance date (test period).
    Green=long, red=short, intensity scales with absolute weight."""
    logs = result.get("all_weights_log", [])
    if not logs:
        print("  [warn] no weights log available; skipping plot_rebalance_weights")
        return

    # Build DataFrame: rows = rebal dates, cols = tickers, values = weights
    rows = []
    for entry in logs:
        row = {"date": entry["date"]}
        row.update(entry["weights"])
        rows.append(row)
    df = pd.DataFrame(rows).set_index("date").fillna(0.0)

    # Keep tickers that ever received nonzero weight, then top-N by absolute weight
    df = df.loc[:, (df.abs().sum(axis=0) > 1e-6)]
    avg_abs = df.abs().mean(axis=0).sort_values(ascending=False)
    keep_tickers = avg_abs.head(max_tickers).index.tolist()
    df = df[keep_tickers]

    fig, ax = plt.subplots(figsize=(15, 7), facecolor=DARK)

    # Custom diverging colormap: red for negative, green for positive
    cmap = LinearSegmentedColormap.from_list("rg_div",
        [RED, "#3a1f1f", DARK, "#1f3a1f", GREEN], N=256)
    vmax = df.abs().values.max()
    im = ax.imshow(df.T.values, aspect="auto", cmap=cmap,
                   vmin=-vmax, vmax=vmax, interpolation="nearest")

    ax.set_yticks(range(len(keep_tickers)))
    ax.set_yticklabels(keep_tickers, fontsize=9)
    # Show every Nth date label to avoid overcrowding
    n_dates = len(df)
    step = max(1, n_dates // 15)
    ax.set_xticks(range(0, n_dates, step))
    ax.set_xticklabels([df.index[i].strftime("%Y-%m") for i in range(0, n_dates, step)],
                       rotation=45, ha="right", fontsize=8)
    ax.set_title(
        f"REBALANCE WEIGHTS HEATMAP  (test period, {n_dates} rebalance dates)  |  "
        "GREEN = long, RED = short",
        color=TEXT, pad=10, fontweight="bold",
    )
    ax.set_xlabel("Rebalance date", color=MUTED)
    ax.set_ylabel("Ticker  (top by |weight|)", color=MUTED)

    cb = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label("Position weight", color=MUTED)
    cb.ax.tick_params(labelcolor=MUTED)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  Saved: {save_path}")
    plt.close()


def plot_optuna(study, save_path="results/mean_rev_final/optuna_analysis.png"):
    """4-panel Optuna search analysis."""
    trials = study.trials_dataframe(attrs=("number", "value", "params", "state"))
    trials = trials[trials["state"] == "COMPLETE"].copy()
    trials.rename(columns={"value": "sortino"}, inplace=True)
    trials.sort_values("sortino", ascending=False, inplace=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=DARK)
    fig.suptitle("OPTUNA TPE SEARCH  //  Objective: Sortino ratio (val)",
                 fontsize=12, fontweight="bold", color=TEXT, y=0.98)

    cmap = LinearSegmentedColormap.from_list("rg", [RED, AMBER, GREEN])
    vmin = trials["sortino"].quantile(0.1)
    vmax = trials["sortino"].quantile(0.9)

    # Panel 1: optimization history
    ax = axes[0, 0]
    trials_sorted = trials.sort_values("number")
    ax.scatter(trials_sorted["number"] + 1, trials_sorted["sortino"],
               c=trials_sorted["sortino"], cmap=cmap, vmin=vmin, vmax=vmax,
               s=30, alpha=0.85)
    running_best = trials_sorted.set_index("number")["sortino"].cummax()
    ax.plot(running_best.index + 1, running_best.values, color=GREEN,
            ls="--", lw=1.5, label="Running best")
    ax.set_xlabel("Trial number", color=MUTED)
    ax.set_ylabel("Sortino ratio", color=MUTED)
    ax.set_title("OPTIMISATION HISTORY", color=TEXT)
    ax.legend(fontsize=9, facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT)
    ax.grid(True, alpha=0.3)

    # Panel 2: top_n_long × gross_short heatmap
    p_n = "params_top_n_long"
    p_gs = "params_gross_short"
    ax = axes[0, 1]
    if p_n in trials.columns and p_gs in trials.columns:
        pivot = trials.groupby([p_n, p_gs])["sortino"].mean().unstack(fill_value=np.nan)
        im = ax.imshow(pivot.values, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{c:.2f}" for c in pivot.columns], fontsize=9)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=9)
        ax.set_xlabel("gross_short", color=MUTED)
        ax.set_ylabel("top_n_long", color=MUTED)
        ax.set_title("top_n_long × gross_short  (mean Sortino)", color=TEXT)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                v = pivot.values[i, j]
                if pd.notna(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=8, color=TEXT)
        plt.colorbar(im, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    # Panel 3: rebal_freq × rsi_period heatmap
    p_r = "params_rebal_freq"
    p_rsi = "params_rsi_period"
    ax = axes[1, 0]
    if p_r in trials.columns and p_rsi in trials.columns:
        pivot = trials.groupby([p_rsi, p_r])["sortino"].mean().unstack(fill_value=np.nan)
        im = ax.imshow(pivot.values, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{c}w" for c in pivot.columns], fontsize=9)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=9)
        ax.set_xlabel("rebal_freq", color=MUTED)
        ax.set_ylabel("rsi_period", color=MUTED)
        ax.set_title("rsi_period × rebal_freq  (mean Sortino)", color=TEXT)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                v = pivot.values[i, j]
                if pd.notna(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=8, color=TEXT)
        plt.colorbar(im, ax=ax, shrink=0.85).ax.tick_params(labelcolor=MUTED)

    # Panel 4: top-20 trials bar chart
    ax = axes[1, 1]
    top20 = trials.head(20)
    bar_colors = [GREEN if s > trials["sortino"].median() else AMBER
                  for s in top20["sortino"]]
    ax.bar(range(len(top20)), top20["sortino"], color=bar_colors, width=0.7)
    labels = []
    for _, r in top20.iterrows():
        labels.append(
            f"R{int(r.get(p_rsi, 0))} {int(r.get(p_r, 0))}w\n"
            f"N{int(r.get(p_n, 0))} g{float(r.get(p_gs, 0)):.2f}"
        )
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(labels, rotation=90, fontsize=6.5)
    ax.set_ylabel("Sortino", color=MUTED)
    ax.set_title("TOP 20 TRIALS BY SORTINO", color=TEXT)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    print(f"  Saved: {save_path}")
    plt.close()


### Main Pipeline

In [12]:
def main():
    print("\n" + "="*60)
    print("  DOW 30 MEAN REVERSION  â€”  ASYMMETRIC LONG-SHORT")
    print("="*60)

    print("\n[1/5]  Downloading 10Y of daily prices + volumes ...")
    prices, volumes = fetch_prices_volumes()
    print(f"       {len(prices)} trading days x {len(prices.columns)} tickers")
    print(f"       {prices.index[0].date()}  ->  {prices.index[-1].date()}")

    print("\n[2/5]  Classifying regimes ...")
    regimes = classify_regimes(prices)
    for s in ["trending", "volatile", "crash"]:
        n = (regimes == s).sum()
        print(f"       {s:10s}: {n:4d} days  ({n/len(regimes)*100:.1f}%)")

    print("\n[3/5]  Running Optuna hyperparameter search ...")
    study, best = run_optuna(prices, volumes, regimes, n_trials=N_TRIALS)

    trials_df = study.trials_dataframe()
    trials_df.to_csv("results/mean_rev_final/optuna_trials.csv", index=False)
    print(f"\n       All trial results  -> results/mean_rev_final/optuna_trials.csv")

    print("\n[4/5]  Final test-period backtest (held out from Optuna) ...")
    feat_best = compute_features_reversal(
        prices, volumes, rsi_period=best["rsi_period"], fwd_days=5)
    result = run_backtest(
        prices, feat_best, regimes,
        rsi_period=best["rsi_period"],
        rebal_freq=best["rebal_freq"],
        lambda_long=best["lambda_long"],
        lambda_short=best["lambda_short"],
        top_n_long=best["top_n_long"],
        top_n_short=best["top_n_short"],
        gross_long=1.0, gross_short=best["gross_short"],
        cap_long=0.12, cap_short=0.06,
        mode="test",
    )

    lc = result["long_leg_curve"]
    sc = result["short_leg_curve"]
    print(f"\n       -- Final metrics --")
    print(f"       Ann. return  : {result['cagr']:.2f}%  (benchmark {result['bench_cagr']:.2f}%)")
    print(f"       Sharpe ratio : {result['sharpe']:.4f}")
    print(f"       Sortino ratio: {result['sortino']:.4f}")
    print(f"       Max drawdown : {result['max_dd']:.2f}%")
    print(f"       Cumulative   : {result['cum_ret']:.1f}%")
    print(f"       Long  leg    : {(lc[-1]/lc[0]-1)*100:+.1f}%  (oversold bouncers)")
    print(f"       Short leg    : {(sc[-1]/sc[0]-1)*100:+.1f}%  (overbought pullbacks)")
    print(f"       Short-kill days: {result['kills_short']}")

    print("\n[5/5]  Generating chart and saving artifacts ...")
    globals()["result"] = result
    plot_validation_performance(prices, feat_best, regimes, best,
                                "results/mean_rev_final/validation_performance.png")
    plot_feature_importance_standalone(result, best,
                                        "results/mean_rev_final/feature_importance.png")
    plot_rebalance_weights(result,
                            "results/mean_rev_final/rebalance_weights.png")
    plot_optuna(study, "results/mean_rev_final/optuna_analysis.png")

    plot_backtest(result, best, "results/mean_rev_final/backtest_summary.png")

    # Save best params + metrics
    with open("results/mean_rev_final/best_params.txt", "w") as f:
        f.write("BEST HYPERPARAMETERS  (Optuna TPE)\n")
        f.write("=" * 40 + "\n")
        for k, v in best.items():
            f.write(f"{k:15s}: {v}\n")
        f.write("\nPERFORMANCE METRICS  (test 2022-2026, held out)\n")
        f.write("=" * 40 + "\n")
        f.write(f"{'sharpe':20s}: {result['sharpe']:.4f}\n")
        f.write(f"{'sortino':20s}: {result['sortino']:.4f}\n")
        f.write(f"{'cagr_%':20s}: {result['cagr']:.2f}\n")
        f.write(f"{'bench_cagr_%':20s}: {result['bench_cagr']:.2f}\n")
        f.write(f"{'max_drawdown_%':20s}: {result['max_dd']:.2f}\n")
        f.write(f"{'cumulative_%':20s}: {result['cum_ret']:.2f}\n")
        f.write(f"{'long_leg_%':20s}: {(lc[-1]/lc[0]-1)*100:.2f}\n")
        f.write(f"{'short_leg_%':20s}: {(sc[-1]/sc[0]-1)*100:.2f}\n")
        f.write(f"{'short_kill_days':20s}: {result['kills_short']}\n")
    print("       Saved: results/mean_rev_final/best_params.txt")

    # Pickle the full result for downstream ICM construction
    result_clean = {k: v for k, v in result.items() if k != "model"}
    with open("results/mean_rev_final/result.pkl", "wb") as f:
        pickle.dump({"result": result_clean, "best": best}, f)
    print("       Saved: results/mean_rev_final/result.pkl")

    print(f"\n{'='*60}\n  Done. All outputs in results/mean_rev_final/\n{'='*60}\n")
    return result, study, prices, regimes, best


if __name__ == "__main__":
    result, study, prices, regimes, best = main()


  DOW 30 MEAN REVERSION  â€”  ASYMMETRIC LONG-SHORT

[1/5]  Downloading 10Y of daily prices + volumes ...


       2526 trading days x 37 tickers
       2016-04-01  ->  2026-04-17

[2/5]  Classifying regimes ...


[I 2026-04-21 20:47:13,645] A new study created in memory with name: mean_rev_asym_ls


       trending  : 1895 days  (75.0%)
       volatile  :  451 days  (17.9%)
       crash     :  180 days  (7.1%)

[3/5]  Running Optuna hyperparameter search ...

  Optuna TPE search  â€”  80 trials (Sortino)



[I 2026-04-21 20:47:46,335] Trial 0 finished with value: 0.5429 and parameters: {'rsi_period': 3, 'rebal_freq': 1, 'lambda_long': 1.9030368381735812, 'lambda_short': 3.027182927734624, 'top_n_long': 20, 'top_n_short': 5, 'gross_short': 0.4}. Best is trial 0 with value: 0.5429.


...

  Trial  80/80  Sortino=+0.815  (best=+0.965)

  BEST TRIAL #77  Sortino = 0.9651
    rsi_period     : 7
    rebal_freq     : 2
    lambda_long    : 0.9116238143505125
    lambda_short   : 3.503138099317926
    top_n_long     : 15
    top_n_short    : 12
    gross_short    : 0.2

       All trial results  -> results/mean_rev_final/optuna_trials.csv

[4/5]  Final test-period backtest (held out from Optuna) ...



       -- Final metrics --
       Ann. return  : 8.49%  (benchmark 10.94%)
       Sharpe ratio : 0.6527
       Sortino ratio: 0.9443
       Max drawdown : -19.97%
       Cumulative   : 41.6%
       Long  leg    : +64.8%  (oversold bouncers)
       Short leg    : -4.1%  (overbought pullbacks)
       Short-kill days: 3

[5/5]  Generating chart and saving artifacts ...


  Saved: results/mean_rev_final/validation_performance.png
  Saved: results/mean_rev_final/feature_importance.png
  [warn] no weights log available; skipping plot_rebalance_weights


  Saved: results/mean_rev_final/optuna_analysis.png


  Saved: results/mean_rev_final/backtest_summary.png
       Saved: results/mean_rev_final/best_params.txt
       Saved: results/mean_rev_final/result.pkl

  Done. All outputs in results/mean_rev_final/



### Diagnostics

In [13]:
# Run after main() completes
strat = np.array(result["strat_curve"])
bench = np.array(result["bench_curve"])
long_c  = np.array(result["long_leg_curve"])
short_c = np.array(result["short_leg_curve"])
dates = pd.DatetimeIndex(result["date_index"])

rets_s = np.diff(strat) / strat[:-1]
rets_b = np.diff(bench) / bench[:-1]
rets_l = np.diff(long_c) / long_c[:-1]
rets_sh = np.diff(short_c) / short_c[:-1]

print(f"Strategy  â€” ann vol: {rets_s.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_s.mean()*252*100:.2f}%")
print(f"Benchmark â€” ann vol: {rets_b.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_b.mean()*252*100:.2f}%")
print(f"Long leg  â€” ann vol: {rets_l.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_l.mean()*252*100:.2f}%")
print(f"Short leg â€” ann vol: {rets_sh.std()*np.sqrt(252)*100:.1f}%   mean daily ret: {rets_sh.mean()*252*100:.2f}%")
print()
print(f"Correlation strat vs bench:   {np.corrcoef(rets_s, rets_b)[0,1]:+.3f}")
print(f"Correlation long  vs bench:   {np.corrcoef(rets_l, rets_b)[0,1]:+.3f}")
print(f"Correlation short vs bench:   {np.corrcoef(rets_sh, rets_b)[0,1]:+.3f}")

df_rets = pd.DataFrame({"strat": rets_s, "bench": rets_b,
                        "long": rets_l, "short": rets_sh}, index=dates[1:])
annual = df_rets.resample("YE").apply(lambda x: (1+x).prod()-1) * 100
print("\nYear-by-year returns (%):")
print(annual.round(2).to_string())

Strategy  â€” ann vol: 14.1%   mean daily ret: 9.17%
Benchmark â€” ann vol: 14.8%   mean daily ret: 11.31%
Long leg  â€” ann vol: 15.5%   mean daily ret: 12.93%
Short leg â€” ann vol: 2.3%   mean daily ret: -0.96%

Correlation strat vs bench:   +0.923
Correlation long  vs bench:   +0.954
Correlation short vs bench:   -0.795

Year-by-year returns (%):
            strat  bench   long  short
2022-12-31 -12.19  -7.61  -9.47  -0.71
2023-12-31  19.64  18.28  23.65  -0.62
2024-12-31  11.02  17.06  16.09  -1.78
2025-12-31  20.59  16.70  23.81  -0.20
2026-12-31   0.77   3.51   2.45  -0.85


In [14]:
# Final weight composition on last rebalance
w = result["weights"]
long_w  = sorted([(t, x) for t, x in w.items() if x > 0], key=lambda x: -x[1])
short_w = sorted([(t, x) for t, x in w.items() if x < 0], key=lambda x: x[1])

print(f"Last rebalance composition:")
print(f"  {len(long_w)} longs totaling  {sum(x for _, x in long_w):+.3f}")
print(f"  {len(short_w)} shorts totaling {sum(x for _, x in short_w):+.3f}")
print(f"  Net exposure              {sum(x for _, x in long_w + short_w):+.3f}")
print(f"  Gross exposure            {sum(abs(x) for _, x in long_w + short_w):+.3f}")
print()
print("LONGS (oversold bouncers):")
for t, x in long_w:
    bar = "#" * int(x * 200)
    print(f"  {t:6s}  {x*100:+5.2f}%  {bar}")
print()
print("SHORTS (overbought pullbacks):")
for t, x in short_w:
    bar = "#" * int(abs(x) * 200)
    print(f"  {t:6s}  {x*100:+5.2f}%  {bar}")

Last rebalance composition:
  15 longs totaling  +1.000
  12 shorts totaling -0.200
  Net exposure              +0.800
  Gross exposure            +1.200

LONGS (oversold bouncers):
  CRM     +7.37%  ##############
  DIS     +6.87%  #############
  AMGN    +6.84%  #############
  MMM     +6.82%  #############
  WMT     +6.74%  #############
  KO      +6.74%  #############
  MSFT    +6.64%  #############
  TRV     +6.61%  #############
  HD      +6.58%  #############
  MRK     +6.53%  #############
  CVX     +6.49%  ############
  AAPL    +6.48%  ############
  MCD     +6.47%  ############
  V       +6.47%  ############
  CSCO    +6.35%  ############

SHORTS (overbought pullbacks):
  NKE     -1.97%  ###
  CAT     -1.73%  ###
  NVDA    -1.69%  ###
  VZ      -1.69%  ###
  HON     -1.68%  ###
  UNH     -1.67%  ###
  AMZN    -1.64%  ###
  IBM     -1.61%  ###
  JNJ     -1.61%  ###
  BA      -1.59%  ###
  SHW     -1.59%  ###
  PG      -1.52%  ###
